# Delta Twin Slice

**Prime Numbers Lab**

This notebook tracks twin-prime-compatible structure inside the residual operator

\[
\Delta = P^{(2)} - P^2.
\]

The tracked residue transitions modulo \(30\) are:

\[
11 \to 13,\qquad 17 \to 19,\qquad 29 \to 1.
\]

This notebook asks:

> Do twin-prime-compatible residue transitions participate in the same higher-order structure observed in \(\Delta\)?

Outputs:

```text
figures/delta_twin_slice_entries.png
figures/delta_twin_slice_vs_controls.png
figures/delta_twin_slice_scaling.png
figures/delta_twin_slice_entries.csv
figures/delta_twin_slice_controls.csv
figures/delta_twin_slice_outputs.zip
```

## 1. Setup

In [ ]:

from pathlib import Path
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path(".")
FIG_DIR = ROOT / "figures"
DATA_DIR = ROOT / "data"

FIG_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

RESIDUES = np.array([1, 7, 11, 13, 17, 19, 23, 29])
RES_IDX = {int(r): i for i, r in enumerate(RESIDUES)}

TWIN_TRANSITIONS = [
    (11, 13),
    (17, 19),
    (29, 1),
]

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})

print("Figure directory:", FIG_DIR.resolve())
print("Data directory:", DATA_DIR.resolve())

## 2. Load repo utilities or use local fallback

In [ ]:

try:
    from src.transitions import primes_mod_30, build_transition_matrix, build_two_step_matrix, RESIDUES
    from src.residual import compute_delta
    RES_IDX = {int(r): i for i, r in enumerate(RESIDUES)}
    print("Loaded utilities from src/")
except Exception as e:
    print("Using local fallback utilities:", repr(e))

    RESIDUES = np.array([1, 7, 11, 13, 17, 19, 23, 29])
    RES_IDX = {int(r): i for i, r in enumerate(RESIDUES)}

    def primes_mod_30(primes):
        return np.array([int(p) % 30 for p in primes if int(p) > 5])

    def build_transition_matrix(seq):
        n = len(RESIDUES)
        P = np.zeros((n, n), dtype=float)

        for i in range(len(seq) - 1):
            a, b = int(seq[i]), int(seq[i + 1])
            if a in RES_IDX and b in RES_IDX:
                P[RES_IDX[a], RES_IDX[b]] += 1

        row_sums = P.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1
        return P / row_sums

    def build_two_step_matrix(seq):
        n = len(RESIDUES)
        P2 = np.zeros((n, n), dtype=float)

        for i in range(len(seq) - 2):
            a, b = int(seq[i]), int(seq[i + 2])
            if a in RES_IDX and b in RES_IDX:
                P2[RES_IDX[a], RES_IDX[b]] += 1

        row_sums = P2.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1
        return P2 / row_sums

    def compute_delta(P, P2):
        return P2 - P @ P

## 3. Load primes

In [ ]:

def load_primes(path=DATA_DIR / "primes.npy", fallback_limit=2_000_000):
    path = Path(path)

    if path.exists():
        print(f"Loading primes from {path}")
        primes = np.load(path)
        print(f"Loaded {len(primes):,} primes.")
        return primes

    print(f"{path} not found.")
    print(f"Generating primes up to {fallback_limit:,} using sympy fallback...")

    try:
        from sympy import primerange
    except ImportError as exc:
        raise ImportError(
            "sympy is required for fallback prime generation. "
            "Install with `pip install sympy`, or provide data/primes.npy."
        ) from exc

    primes = np.array(list(primerange(2, fallback_limit)), dtype=np.int64)
    print(f"Generated {len(primes):,} primes.")
    return primes

primes = load_primes()
print("First primes:", primes[:10])
print("Last primes:", primes[-5:])

## 4. Core functions

In [ ]:

def operators_from_primes(primes_subset):
    seq = primes_mod_30(primes_subset)
    P = build_transition_matrix(seq)
    P2 = build_two_step_matrix(seq)
    delta = compute_delta(P, P2)
    return seq, P, P2, delta

def transition_entry(M, src, dst):
    return float(M[RES_IDX[int(src)], RES_IDX[int(dst)]])

def twin_delta_entries(delta):
    return {
        f"{src}->{dst}": transition_entry(delta, src, dst)
        for src, dst in TWIN_TRANSITIONS
    }

def twin_delta_norm(delta):
    values = np.array(list(twin_delta_entries(delta).values()), dtype=float)
    return float(np.linalg.norm(values))

def fro_norm(M):
    return float(np.linalg.norm(M, ord="fro"))

def sample_sizes(n_total):
    candidates = [
        1_000,
        3_000,
        10_000,
        30_000,
        100_000,
        300_000,
        1_000_000,
        3_000_000,
        10_000_000,
    ]
    sizes = [n for n in candidates if n <= n_total]
    if not sizes:
        sizes = [n_total]
    return sizes

## 5. Compute \(\Delta\) twin slice for the largest sample

In [ ]:

N_MAX = len(primes)

seq, P, P2, delta = operators_from_primes(primes[:N_MAX])

entry_dict = twin_delta_entries(delta)
entry_df = pd.DataFrame({
    "transition": list(entry_dict.keys()),
    "delta_entry": list(entry_dict.values()),
})

entry_df

## 6. Plot \(\Delta\) entries for twin-prime-compatible transitions

In [ ]:

fig, ax = plt.subplots(figsize=(7, 4.6))

labels = entry_df["transition"].tolist()
values = entry_df["delta_entry"].to_numpy()

ax.axhline(0, linewidth=1)
ax.bar(labels, values)
ax.set_ylabel(r"$\Delta$ entry")
ax.set_title(r"Twin-prime-compatible entries in $\Delta = P^{(2)} - P^2$")
fig.tight_layout()

entries_png = FIG_DIR / "delta_twin_slice_entries.png"
fig.savefig(entries_png, dpi=220)
plt.show()

entries_csv = FIG_DIR / "delta_twin_slice_entries.csv"
entry_df.to_csv(entries_csv, index=False)

print("Saved:", entries_png)
print("Saved:", entries_csv)

## 7. Compare twin slice against controls

In [ ]:

def make_control_sequences(seq, P, n_controls=100, block_size=50, seed=42):
    rng = np.random.default_rng(seed)
    controls = {"iid": [], "markov": [], "block": []}

    n_states = P.shape[0]

    for _ in range(n_controls):
        # iid shuffle of residue sequence
        controls["iid"].append(rng.permutation(seq))

        # Markov control in residue-index space, mapped back to residue labels
        idx_seq = [rng.integers(n_states)]
        for _ in range(len(seq) - 1):
            idx_seq.append(rng.choice(n_states, p=P[idx_seq[-1]]))
        controls["markov"].append(RESIDUES[np.array(idx_seq)])

        # Block shuffle
        blocks = [seq[i:i + block_size] for i in range(0, len(seq), block_size)]
        rng.shuffle(blocks)
        controls["block"].append(np.concatenate(blocks))

    return controls

def delta_from_sequence(seq):
    P = build_transition_matrix(seq)
    P2 = build_two_step_matrix(seq)
    return compute_delta(P, P2)

N_CONTROL = min(len(primes), 300_000)
print(f"Using N={N_CONTROL:,} for control comparison.")

seq_c, P_c, P2_c, delta_c = operators_from_primes(primes[:N_CONTROL])

controls = make_control_sequences(seq_c, P_c, n_controls=100, block_size=50)

rows = []
real_entries = twin_delta_entries(delta_c)
real_slice_norm = twin_delta_norm(delta_c)

for transition, value in real_entries.items():
    rows.append({
        "control": "prime",
        "transition": transition,
        "delta_entry": value,
        "slice_norm": real_slice_norm,
    })

for control_name, seqs in controls.items():
    for s in seqs:
        d = delta_from_sequence(s)
        entries = twin_delta_entries(d)
        norm = twin_delta_norm(d)
        for transition, value in entries.items():
            rows.append({
                "control": control_name,
                "transition": transition,
                "delta_entry": value,
                "slice_norm": norm,
            })

controls_df = pd.DataFrame(rows)
controls_df.head()

## 8. Plot \(\Delta\) twin slice versus controls

In [ ]:

summary = (
    controls_df
    .groupby(["control", "transition"])["delta_entry"]
    .agg(["mean", "std"])
    .reset_index()
)

transitions = [f"{a}->{b}" for a, b in TWIN_TRANSITIONS]
control_names = ["iid", "markov", "block"]

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)

for ax, transition in zip(axes, transitions):
    real_value = controls_df[
        (controls_df["control"] == "prime") &
        (controls_df["transition"] == transition)
    ]["delta_entry"].iloc[0]

    means = []
    stds = []

    for control in control_names:
        vals = controls_df[
            (controls_df["control"] == control) &
            (controls_df["transition"] == transition)
        ]["delta_entry"]
        means.append(vals.mean())
        stds.append(vals.std())

    ax.axhline(0, linewidth=1)
    ax.bar(["prime"], [real_value], label="prime")
    ax.bar(control_names, means, yerr=stds, capsize=4, alpha=0.75)
    ax.set_title(transition)
    ax.set_ylabel(r"$\Delta$ entry")

fig.suptitle(r"Twin-prime-compatible $\Delta$ entries vs controls", y=1.03)
fig.tight_layout()

controls_png = FIG_DIR / "delta_twin_slice_vs_controls.png"
fig.savefig(controls_png, dpi=220, bbox_inches="tight")
plt.show()

controls_csv = FIG_DIR / "delta_twin_slice_controls.csv"
controls_df.to_csv(controls_csv, index=False)

print("Saved:", controls_png)
print("Saved:", controls_csv)

## 9. Scaling over \(N\)

In [ ]:

sizes = sample_sizes(len(primes))
print("Scaling sample sizes:", sizes)

scale_rows = []

for n in sizes:
    seq_n, P_n, P2_n, delta_n = operators_from_primes(primes[:n])
    entries = twin_delta_entries(delta_n)
    slice_norm = twin_delta_norm(delta_n)
    total_norm = fro_norm(delta_n)

    row = {
        "N": n,
        "slice_norm": slice_norm,
        "delta_fro_norm": total_norm,
        "slice_norm_ratio": slice_norm / (total_norm + 1e-12),
    }
    row.update(entries)
    scale_rows.append(row)

scale_df = pd.DataFrame(scale_rows)
scale_df

## 10. Plot scaling

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

# Left: individual entries
for transition in [f"{a}->{b}" for a, b in TWIN_TRANSITIONS]:
    axes[0].plot(scale_df["N"], scale_df[transition], marker="o", linewidth=2, label=transition)

axes[0].axhline(0, linewidth=1)
axes[0].set_xscale("log")
axes[0].set_xlabel("Number of primes used")
axes[0].set_ylabel(r"$\Delta$ entry")
axes[0].set_title(r"Twin-compatible entries in $\Delta$")
axes[0].legend(title="Transition")

# Right: slice norm ratio
axes[1].plot(scale_df["N"], scale_df["slice_norm_ratio"], marker="o", linewidth=2)
axes[1].set_xscale("log")
axes[1].set_xlabel("Number of primes used")
axes[1].set_ylabel(r"$\|\Delta_{\mathrm{twin}}\| / \|\Delta\|_F$")
axes[1].set_title("Twin slice share of residual norm")

fig.tight_layout()

scaling_png = FIG_DIR / "delta_twin_slice_scaling.png"
fig.savefig(scaling_png, dpi=220)
plt.show()

scaling_csv = FIG_DIR / "delta_twin_slice_scaling.csv"
scale_df.to_csv(scaling_csv, index=False)

print("Saved:", scaling_png)
print("Saved:", scaling_csv)

## 11. Interpretation

This notebook tracks twin-prime-compatible entries inside the residual operator

\[
\Delta = P^{(2)} - P^2.
\]

The entries

\[
11 \to 13,\qquad 17 \to 19,\qquad 29 \to 1
\]

are the residue transitions modulo \(30\) compatible with twin prime gaps.

This notebook does **not** address the asymptotic twin prime conjecture.

It provides a measurable slice of \(\Delta\) tied to twin-prime-compatible residue structure.

## 12. Package outputs

In [ ]:

zip_path = FIG_DIR / "delta_twin_slice_outputs.zip"

output_files = [
    FIG_DIR / "delta_twin_slice_entries.png",
    FIG_DIR / "delta_twin_slice_vs_controls.png",
    FIG_DIR / "delta_twin_slice_scaling.png",
    FIG_DIR / "delta_twin_slice_entries.csv",
    FIG_DIR / "delta_twin_slice_controls.csv",
    FIG_DIR / "delta_twin_slice_scaling.csv",
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in output_files:
        if path.exists():
            z.write(path, arcname=path.name)

print("Saved:", zip_path)
print("Contents:")
with zipfile.ZipFile(zip_path, "r") as z:
    for name in z.namelist():
        print(" -", name)

## 13. Optional Colab download

In [ ]:

try:
    from google.colab import files
    files.download(str(FIG_DIR / "delta_twin_slice_outputs.zip"))
except ImportError:
    print("Download only works in Google Colab.")
    print("Zip saved locally at:")
    print(FIG_DIR / "delta_twin_slice_outputs.zip")